# AGRICULTURAL CROP CONDITION ANALYSIS
### Clustering-Based Crop Suitability Recommendation System
**Academic Project | AI & Data Science**

**Objective:**
Develop an unsupervised machine learning pipeline using K-Means Clustering and Principal Component Analysis (PCA) to group agricultural crops according to their multi-dimensional environmental requirements and recommend suitable crops based on user-provided environmental conditions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import joblib

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries imported successfully.')

## 1. Dataset Loading and Schema Inspection
We load the agricultural crop environmental requirement dataset from `data/crop_environment.csv`. The dataset features 8 key agronomic parameters:
- **Temperature (°C)**
- **Rainfall (mm)**
- **Humidity (%)**
- **Soil Moisture (%)**
- **Soil pH**
- **Nitrogen (N) (kg/ha)**
- **Phosphorus (P) (kg/ha)**
- **Potassium (K) (kg/ha)**

In [ ]:
data_path = '../data/crop_environment.csv' if os.path.exists('../data/crop_environment.csv') else 'data/crop_environment.csv'
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
print(f'Unique Crop Varieties: {df["crop_name"].nunique()}')
display(df.head())
display(df.info())

## 2. Exploratory Data Analysis (EDA)
Let us examine descriptive statistics, missing values, and the Pearson correlation matrix.

In [ ]:
features = ['temperature', 'rainfall', 'humidity', 'soil_moisture', 'soil_ph', 'nitrogen', 'phosphorus', 'potassium']
display(df[features].describe().round(2))

# Correlation Heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(df[features].corr(), annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title('Pearson Correlation Matrix of Environmental Features', fontsize=14, fontweight='bold')
plt.show()

## 3. Feature Standardization (StandardScaler)\nBecause environmental features operate on vastly different physical scales (Rainfall: 300–2500 mm vs. Soil pH: 5–8), we must standardize all features to zero mean ($\mu=0$) and unit variance ($\sigma=1$). Otherwise, high-magnitude features would dominate Euclidean distance metrics.

In [ ]:
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Features standardized successfully. Mean:', np.round(X_scaled.mean(axis=0), 2))
print('Standard Deviation:', np.round(X_scaled.std(axis=0), 2))

## 4. Determining Optimal Number of Clusters (K)
We evaluate K values from 2 to 10 using:
1. **Elbow Method (Inertia / Within-Cluster Sum of Squares - WCSS)**
2. **Silhouette Coefficient (Cohesion vs. Separation Quality)**

In [ ]:
k_range = list(range(2, 11))
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', color=color, fontsize=12)
ax1.plot(k_range, inertias, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Silhouette Score', color=color, fontsize=12)
ax2.plot(k_range, silhouette_scores, marker='s', color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Cluster Evaluation: Elbow Method & Silhouette Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(silhouette_scores)]
print(f'Optimal K by Silhouette Score: {best_k} (Score: {max(silhouette_scores):.4f})')

## 5. Training Final K-Means Model
We fit the K-Means algorithm using the optimal cluster count $K=4$.

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster_id'] = kmeans.fit_predict(X_scaled)
overall_silhouette = silhouette_score(X_scaled, df['cluster_id'])
print(f'Final Model Silhouette Score: {overall_silhouette:.4f}')

## 6. Dimensionality Reduction with PCA for 2D Visualization
Because our dataset has 8 dimensions, we project it onto the first two principal components.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['pca_1'] = X_pca[:, 0]
df['pca_2'] = X_pca[:, 1]

print(f'Explained Variance Ratio: {pca.explained_variance_ratio_}')
print(f'Total Variance Explained: {pca.explained_variance_ratio_.sum()*100:.2f}%')

plt.figure(figsize=(11, 7))
scatter = sns.scatterplot(
    data=df,
    x='pca_1',
    y='pca_2',
    hue='cluster_id',
    palette='Set2',
    s=80,
    alpha=0.85
)
plt.title('2D PCA Projection of Crop Environmental Clusters', fontsize=14, fontweight='bold')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)')
plt.legend(title='Cluster ID')
plt.show()

## 7. Dynamic Cluster Interpretation & Profiling
We calculate the mean environmental attributes for each cluster and dynamically interpret the agronomic profiles.

In [ ]:
cluster_means = df.groupby('cluster_id')[features].mean().round(2)
display(cluster_means)

for cid in range(optimal_k):
    crops = sorted(df[df['cluster_id'] == cid]['crop_name'].unique())
    print(f'=== Cluster {cid} ({len(crops)} crops) ===')
    print('Crops:', ', '.join(crops))
    print('Key Means:', dict(cluster_means.loc[cid][['temperature', 'rainfall', 'soil_moisture', 'soil_ph']]))
    print()

## 8. Crop Recommendation Simulation
Given user input conditions, the pipeline:
1. Scales the query vector with `StandardScaler`
2. Predicts the closest cluster centroid
3. Computes Euclidean distance to crop centroids
4. Converts distance into a 0–100% Suitability Score
5. Ranks top crops

In [ ]:
# Simulated user input (e.g. wetland condition)
user_input = np.array([[28.0, 1750.0, 85.0, 82.0, 6.2, 85.0, 45.0, 45.0]])
user_scaled = scaler.transform(user_input)
pred_cluster = kmeans.predict(user_scaled)[0]
print(f'Predicted Cluster: {pred_cluster}')

# Compute distance to each crop's mean profile in scaled space
crop_profiles = df.groupby('crop_name')[features].mean()
crop_scaled = scaler.transform(crop_profiles.values)
distances = np.linalg.norm(crop_scaled - user_scaled, axis=1)

crop_scores = pd.DataFrame({
    'crop': crop_profiles.index,
    'distance': distances,
    'suitability_pct': np.round(100.0 * np.exp(-(distances ** 2) / (2.0 * (2.0 ** 2))), 1)
}).sort_values(by='suitability_pct', ascending=False)

print('Top Recommended Crops:')
display(crop_scores.head(5))

## 9. Model Serialization
Models are saved using Joblib so the Streamlit application can load them without retraining.

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(kmeans, '../models/kmeans_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(pca, '../models/pca_model.pkl')
print('Models persisted to models/ directory successfully.')